# Figure 5: reference-based TLS state classification

This notebook provides the reproducible TLS classification workflow used for the
spatial lung cohort. It performs five operations:

1. sorts normal naive and germinal-center B cells by Monocle3 pseudotime;
2. constructs 100 equal-cell-count reference pseudocells;
3. aligns spatial B cells to the reference using Spearman correlations and two anchors;
4. calls Mfuzz with a fixed cluster count and random seed;
5. integrates the trajectory group with the sample-specific CR2 mature-TLS rule.

The monotonic pseudotime checks are mandatory. The notebook stops if reference cells or
pseudocells are not ordered. The labels `Conforming` and `Deviating` are retained for
compatibility with the source-data tables and correspond to trajectory-aligned and
trajectory-divergent states, respectively.


In [ ]:
from pathlib import Path
from multiprocessing.pool import ThreadPool
import os
import subprocess
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from scipy.stats import spearmanr
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)


def env_path(name, default):
    value = Path(os.environ.get(name, str(default))).expanduser()
    return value if value.is_absolute() else (PROJECT_ROOT / value).resolve()


# Set TLS_PROJECT_ROOT to the directory containing this notebook and its data folder.
PROJECT_ROOT = Path(os.environ.get("TLS_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_ROOT = env_path(
    "TLS_CLASSIFICATION_OUTPUT",
    PROJECT_ROOT / "results" / "Fig5_TLS_classification",
)

SPATIAL_H5AD = env_path("TLS_SPATIAL_H5AD", DATA_DIR / "spatial_lung.h5ad")
REFERENCE_H5AD = env_path(
    "TLS_REFERENCE_H5AD",
    DATA_DIR / "normal_LN_lung_naive_GCB_counts.h5ad",
)
PSEUDOTIME_TSV = env_path(
    "TLS_PSEUDOTIME_TSV",
    DATA_DIR / "NaiveB_GC_trajectory_pseudotime.tsv",
)
MFUZZ_SCRIPT = env_path(
    "TLS_MFUZZ_SCRIPT",
    PROJECT_ROOT / "R" / "main_figures" / "Fig5_TLS_classification_mfuzz.R",
)
RSCRIPT_BIN = os.environ.get("RSCRIPT_BIN", "Rscript")


ALIGNMENT_DIR = OUTPUT_ROOT / "alignment"
MFUZZ_DIR = OUTPUT_ROOT / "mfuzz"
FINAL_DIR = OUTPUT_ROOT / "final"
for directory in (ALIGNMENT_DIR, MFUZZ_DIR, FINAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Spatial metadata
TLS_COL = "tls_degrow_id"
SAMPLE_COL = "sample"
ALIGNMENT_B_CELL_COL = "celltype_3_ZZM"
ALIGNMENT_B_CELL_LABELS = ("B cells",)
CR2_B_CELL_COL = "celltype_1_ZZM"
CR2_B_CELL_LABEL = "B cells"

# Alignment parameters
N_PSEUDOCELLS = 100
TOP_K = 2
NOMINAL_P_THRESHOLD = 0.05
N_JOBS = min(20, os.cpu_count() or 1)
IQR_WHISKER = 1.5

# Mfuzz parameters
MFUZZ_CLUSTERS = 10
MFUZZ_SEED = 1

# Mature-TLS parameters
CR2_GENE = "CR2"
CR2_POSITIVE_QUANTILE = 0.75
MIN_B_CELLS_PER_TLS = 10
MIN_CR2_POSITIVE_FRACTION = 0.10
CPM_SCALE = 1e6

REUSE_EXISTING_CORRELATIONS = os.environ.get("TLS_REUSE_CORRELATIONS", "0") == "1"
WRITE_ANNOTATED_H5AD = os.environ.get("TLS_WRITE_H5AD", "0") == "1"

CORRELATION_PATH = ALIGNMENT_DIR / "spatial_B_to_reference_spearman.tsv.gz"
PSEUDOCELL_QC_PATH = ALIGNMENT_DIR / "reference_pseudocell_qc.tsv"
ALIGNMENT_SUM_PATH = ALIGNMENT_DIR / "TLS_alignment_top2_sum.tsv"
ALIGNMENT_COUNT_PATH = ALIGNMENT_DIR / "TLS_alignment_top2_count.tsv"
PRE_MFUZZ_PATH = ALIGNMENT_DIR / "pre_mfuzz.tsv"
ALIGNMENT_QC_PATH = ALIGNMENT_DIR / "TLS_alignment_qc.tsv"
UNASSIGNED_PATH = ALIGNMENT_DIR / "TLS_unassigned_before_mfuzz.tsv"
CR2_METRICS_PATH = FINAL_DIR / "TLS_CR2_metrics.tsv"
FINAL_CLASSIFICATION_PATH = FINAL_DIR / "TLS_final_classification.tsv"
FINAL_H5AD_PATH = FINAL_DIR / "spatial_with_TLS_classification.h5ad"

print(f"Project root: {PROJECT_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Spatial input: {SPATIAL_H5AD}")
print(f"Reference input: {REFERENCE_H5AD}")
print(f"Pseudotime input: {PSEUDOTIME_TSV}")


In [ ]:
def require_columns(frame, columns, label):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise KeyError(f"{label} is missing required columns: {missing}")


def valid_tls_mask(obs):
    values = obs[TLS_COL].astype("string").str.strip()
    return values.notna() & values.ne("") & values.str.lower().ne("none")


def raw_or_x(adata):
    result = adata.raw.to_adata() if adata.raw is not None else adata.copy()
    result.var_names_make_unique()
    return result


def normalize_log1p(adata):
    result = adata.copy()
    sc.pp.normalize_total(result, target_sum=1e4)
    result.uns.pop("log1p", None)
    sc.pp.log1p(result)
    return result


def dense_array(matrix):
    return matrix.toarray() if sparse.issparse(matrix) else np.asarray(matrix)


def read_ordered_pseudotime(path):
    frame = pd.read_csv(path, sep="	")
    frame = frame.loc[:, ~frame.columns.astype(str).str.startswith("Unnamed")]
    require_columns(frame, ["cell_id", "pseudotime"], "pseudotime table")
    frame = frame[["cell_id", "pseudotime"]].copy()
    frame["cell_id"] = frame["cell_id"].astype(str).str.strip()
    frame["pseudotime"] = pd.to_numeric(frame["pseudotime"], errors="coerce")
    frame = frame.dropna().drop_duplicates("cell_id", keep="first")
    frame = frame.sort_values(
        ["pseudotime", "cell_id"],
        kind="mergesort",
    ).set_index("cell_id")
    if not frame["pseudotime"].is_monotonic_increasing:
        raise RuntimeError("Reference pseudotime is not monotonic after sorting.")
    return frame


def choose_mode(values):
    values = values.dropna().astype(str)
    if values.empty:
        return pd.NA
    mode = values.mode()
    return mode.iloc[0] if not mode.empty else values.iloc[0]


def matrix_and_genes(adata):
    if adata.raw is not None:
        return adata.raw.X, adata.raw.var_names
    return adata.X, adata.var_names


def gene_vector(matrix, var_names, gene):
    lookup = {str(name).upper(): index for index, name in enumerate(var_names)}
    if gene.upper() not in lookup:
        raise KeyError(f"Gene {gene} was not found in the expression matrix.")
    index = lookup[gene.upper()]
    if sparse.issparse(matrix):
        return matrix[:, index].toarray().ravel().astype(float)
    return np.asarray(matrix[:, index], dtype=float).ravel()


## 1. Load spatial B cells and the reference trajectory

Only non-empty TLS identifiers are retained. Reference cells are selected by cell ID and
then reordered by ascending Monocle3 pseudotime before any pseudocell is constructed.


In [ ]:
for path in (SPATIAL_H5AD, REFERENCE_H5AD, PSEUDOTIME_TSV, MFUZZ_SCRIPT):
    if not path.exists():
        raise FileNotFoundError(path)

pseudotime = read_ordered_pseudotime(PSEUDOTIME_TSV)

spatial_backed = sc.read_h5ad(SPATIAL_H5AD, backed="r")
require_columns(
    spatial_backed.obs,
    [TLS_COL, SAMPLE_COL, ALIGNMENT_B_CELL_COL, CR2_B_CELL_COL],
    "spatial adata.obs",
)
tls_mask = valid_tls_mask(spatial_backed.obs)
all_tls_ids = pd.Index(
    spatial_backed.obs.loc[tls_mask, TLS_COL].astype(str).drop_duplicates(),
    name="tls_id",
)
alignment_b_mask = (
    tls_mask
    & spatial_backed.obs[ALIGNMENT_B_CELL_COL]
    .astype(str)
    .isin(ALIGNMENT_B_CELL_LABELS)
)
spatial_b = raw_or_x(spatial_backed[alignment_b_mask].to_memory())
spatial_backed.file.close()

reference_backed = sc.read_h5ad(REFERENCE_H5AD, backed="r")
ordered_reference_ids = pseudotime.index[
    pseudotime.index.isin(reference_backed.obs_names)
]
if len(ordered_reference_ids) < N_PSEUDOCELLS:
    reference_backed.file.close()
    raise ValueError("Fewer reference cells than requested pseudocells were recovered.")
reference = raw_or_x(reference_backed[ordered_reference_ids].to_memory())
reference_backed.file.close()

if spatial_b.n_obs == 0:
    raise ValueError("No spatial B cells were found inside TLSs.")

print(f"Independent TLS-like structures: {len(all_tls_ids)}")
print(f"Spatial B cells used for alignment: {spatial_b.n_obs}")
print(f"Ordered reference B cells: {reference.n_obs}")


## 2. Construct ordered reference pseudocells

The ordered cells are divided into 100 approximately equal-sized bins. Mean expression
within each bin defines one reference pseudocell. Both cell-level and pseudocell-level
monotonicity are checked before alignment.


In [ ]:
spatial_b = normalize_log1p(spatial_b)
reference = normalize_log1p(reference)

common_genes = spatial_b.var_names.intersection(reference.var_names, sort=False)
if common_genes.empty:
    raise ValueError("No common genes were found between spatial and reference B cells.")

spatial_b = spatial_b[:, common_genes].copy()
reference = reference[:, common_genes].copy()
reference = reference[pseudotime.index[pseudotime.index.isin(reference.obs_names)]].copy()

reference_expression = pd.DataFrame(
    dense_array(reference.X),
    index=reference.obs_names,
    columns=common_genes,
)
spatial_expression = pd.DataFrame(
    dense_array(spatial_b.X),
    index=spatial_b.obs_names,
    columns=common_genes,
)

split_indices = np.array_split(np.arange(reference.n_obs), N_PSEUDOCELLS)
bin_id = np.empty(reference.n_obs, dtype=int)
for position, indices in enumerate(split_indices, start=1):
    bin_id[indices] = position

pseudocell_expression = reference_expression.groupby(bin_id, sort=True).mean()
pseudocell_expression.index.name = "reference_position"

pseudocell_qc = pd.DataFrame(
    {
        "n_reference_cells": pd.Series(bin_id).value_counts().sort_index(),
        "min_pseudotime": pseudotime.loc[reference.obs_names, "pseudotime"].groupby(bin_id).min(),
        "mean_pseudotime": pseudotime.loc[reference.obs_names, "pseudotime"].groupby(bin_id).mean(),
        "max_pseudotime": pseudotime.loc[reference.obs_names, "pseudotime"].groupby(bin_id).max(),
    }
)
pseudocell_qc.index.name = "reference_position"

if pseudocell_expression.shape[0] != N_PSEUDOCELLS:
    raise RuntimeError("The reference did not produce exactly 100 pseudocells.")
if not pseudocell_qc["mean_pseudotime"].is_monotonic_increasing:
    raise RuntimeError("Pseudocell mean pseudotime is not monotonic.")

pseudocell_qc.to_csv(PSEUDOCELL_QC_PATH, sep="	")
print(f"Common genes used for correlations: {len(common_genes):,}")
print(pseudocell_qc.iloc[[0, -1]])

fig, ax = plt.subplots(figsize=(6.0, 3.6))
ax.plot(pseudocell_qc.index, pseudocell_qc["mean_pseudotime"], color="#2369A1")
ax.set(xlabel="Reference pseudocell position", ylabel="Mean Monocle3 pseudotime")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()


## 3. Align spatial B cells to the reference

Spearman correlations are calculated over genes present in both matrices. Genes with zero
summed expression in a cell-pseudocell pair are omitted. Correlations with nominal
`P >= 0.05` are set to zero. This P-value is a matching filter, not a group-level test.


In [ ]:
pseudocell_array = pseudocell_expression.to_numpy(dtype=float, copy=False)
spatial_array = spatial_expression.to_numpy(dtype=float, copy=False)


def correlate_one_spatial_cell(cell_values):
    result = np.zeros(pseudocell_array.shape[0], dtype=float)
    for index, reference_values in enumerate(pseudocell_array):
        keep = (cell_values + reference_values) > 0
        if keep.sum() < 3:
            continue
        rho, p_value = spearmanr(cell_values[keep], reference_values[keep])
        if np.isfinite(rho) and np.isfinite(p_value) and p_value < NOMINAL_P_THRESHOLD:
            result[index] = float(rho)
    return result


if REUSE_EXISTING_CORRELATIONS and CORRELATION_PATH.exists():
    cell_reference_correlation = pd.read_csv(
        CORRELATION_PATH,
        sep="	",
        index_col=0,
    )
    cell_reference_correlation.columns = cell_reference_correlation.columns.astype(int)
    expected_cells = pd.Index(spatial_expression.index)
    if not cell_reference_correlation.index.equals(expected_cells):
        raise RuntimeError("Cached correlations do not match the current spatial B cells.")
else:
    if N_JOBS == 1:
        rows = [correlate_one_spatial_cell(row) for row in tqdm(spatial_array)]
    else:
        with ThreadPool(processes=N_JOBS) as pool:
            rows = list(
                tqdm(
                    pool.imap(correlate_one_spatial_cell, spatial_array, chunksize=4),
                    total=spatial_array.shape[0],
                )
            )
    cell_reference_correlation = pd.DataFrame(
        np.vstack(rows),
        index=spatial_expression.index,
        columns=pseudocell_expression.index.astype(int),
    )
    cell_reference_correlation.to_csv(
        CORRELATION_PATH,
        sep="	",
        compression="gzip",
    )

print(f"Cell-reference correlation matrix: {cell_reference_correlation.shape}")
print(f"Saved correlations: {CORRELATION_PATH}")


## 4. Build TLS-level alignment profiles

For each spatial B cell, the two strongest positive reference correlations are retained.
Each coefficient contributes separately to its corresponding reference position. The
coefficients are summed across B cells within each TLS-like structure.


In [ ]:
alignment_sum = pd.DataFrame(
    0.0,
    index=all_tls_ids,
    columns=pseudocell_expression.index.astype(int),
)
alignment_count = pd.DataFrame(
    0,
    index=all_tls_ids,
    columns=pseudocell_expression.index.astype(int),
    dtype=int,
)
contributing_cells = pd.Series(0, index=all_tls_ids, dtype=int)
cell_to_tls = spatial_b.obs[TLS_COL].astype(str)

for cell_id, correlations in cell_reference_correlation.iterrows():
    top = correlations[correlations > 0].nlargest(TOP_K)
    if top.empty:
        continue
    tls_id = cell_to_tls.loc[cell_id]
    contributing_cells.loc[tls_id] += 1
    for position, rho in top.items():
        alignment_sum.loc[tls_id, int(position)] += float(rho)
        alignment_count.loc[tls_id, int(position)] += 1

alignment_sum.to_csv(ALIGNMENT_SUM_PATH, sep="	")
alignment_count.to_csv(ALIGNMENT_COUNT_PATH, sep="	")


def positions_within_iqr(positions, whisker=1.5):
    positions = np.asarray(sorted(positions), dtype=int)
    if positions.size < 2:
        return positions
    q1, q3 = np.quantile(positions, [0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - whisker * iqr
    upper = q3 + whisker * iqr
    return positions[(positions >= lower) & (positions <= upper)]


profiles = {}
qc_rows = []
for tls_id in alignment_sum.index:
    all_positions = alignment_sum.columns[
        alignment_sum.loc[tls_id] > 0
    ].to_numpy(dtype=int)
    retained_positions = positions_within_iqr(all_positions, IQR_WHISKER)
    qc = {
        "tls_id": tls_id,
        "n_spatial_B_cells": int((cell_to_tls == tls_id).sum()),
        "n_contributing_B_cells": int(contributing_cells.loc[tls_id]),
        "n_top2_assignments": int(alignment_count.loc[tls_id].sum()),
        "n_nonzero_reference_positions": int(len(all_positions)),
        "n_retained_reference_positions": int(len(retained_positions)),
    }

    if len(retained_positions) < 2:
        qc.update({"slope": np.nan, "intercept": np.nan, "status": "unassigned"})
        qc_rows.append(qc)
        continue

    values = alignment_sum.loc[tls_id, retained_positions].to_numpy(dtype=float)
    slope, intercept = np.polyfit(np.arange(len(values), dtype=float), values, 1)
    profiles[tls_id] = intercept + slope * np.arange(10, dtype=float)
    qc.update({"slope": slope, "intercept": intercept, "status": "included_in_mfuzz"})
    qc_rows.append(qc)

pre_mfuzz = pd.DataFrame.from_dict(profiles, orient="index", columns=range(10))
pre_mfuzz.index.name = "tls_id"
alignment_qc = pd.DataFrame(qc_rows).set_index("tls_id")
unassigned = alignment_qc[alignment_qc["status"] == "unassigned"]

pre_mfuzz.to_csv(PRE_MFUZZ_PATH, sep="	")
alignment_qc.to_csv(ALIGNMENT_QC_PATH, sep="	")
unassigned.to_csv(UNASSIGNED_PATH, sep="	")

print(f"TLS profiles exported to Mfuzz: {len(pre_mfuzz)}")
print(f"TLSs unassigned before Mfuzz: {len(unassigned)}")
print(f"Saved Mfuzz input: {PRE_MFUZZ_PATH}")


## 5. Run Mfuzz with fixed parameters

The companion R script runs Mfuzz with ten clusters and random seed 1. Cluster centers
with non-negative Spearman correlation to the ordered profile index are labeled
`Conforming`; centers with negative correlation are labeled `Deviating`.


In [ ]:
r_environment = os.environ.copy()
r_environment["TLS_MFUZZ_CLUSTERS"] = str(MFUZZ_CLUSTERS)
r_environment["TLS_MFUZZ_SEED"] = str(MFUZZ_SEED)

completed = subprocess.run(
    [RSCRIPT_BIN, str(MFUZZ_SCRIPT), str(PRE_MFUZZ_PATH), str(MFUZZ_DIR)],
    check=True,
    text=True,
    capture_output=True,
    env=r_environment,
)
print(completed.stdout)
print(completed.stderr)

trajectory_groups_path = MFUZZ_DIR / "trajectory_groups.tsv"
trajectory = pd.read_csv(trajectory_groups_path, sep="	", index_col=0)
require_columns(trajectory, ["Group"], "Mfuzz trajectory table")
trajectory = trajectory.loc[~trajectory.index.duplicated(keep="first")]
print("Trajectory-group counts:")
print(trajectory["Group"].value_counts())


## 6. Integrate the CR2 mature-TLS rule

CR2 counts are converted to counts per million. A B cell is CR2-positive when its value
exceeds the 75th percentile among all B cells from the same sample. A structure is labeled
`Mature` when it contains at least ten B cells and at least 10% are CR2-positive.


In [ ]:
adata = sc.read_h5ad(SPATIAL_H5AD)
tls_values = adata.obs[TLS_COL].astype("string").str.strip()
tls_valid = valid_tls_mask(adata.obs)

trajectory_lookup = trajectory["Group"].astype(str)
trajectory_labels = tls_values.map(trajectory_lookup).astype("string")
trajectory_labels.loc[~tls_valid] = pd.NA

expression, var_names = matrix_and_genes(adata)
cr2_counts = gene_vector(expression, var_names, CR2_GENE)
library_size = np.asarray(expression.sum(axis=1)).ravel().astype(float)
cr2_cpm = cr2_counts / np.clip(library_size, 1.0, None) * CPM_SCALE

b_values = adata.obs[CR2_B_CELL_COL].astype("string").fillna("").str.strip()
is_b_cell = b_values.str.lower().eq(CR2_B_CELL_LABEL.lower()).to_numpy(dtype=bool)
sample_values = adata.obs[SAMPLE_COL].astype("string").fillna("NA")

global_b_values = cr2_cpm[is_b_cell]
global_threshold = (
    float(np.quantile(global_b_values, CR2_POSITIVE_QUANTILE))
    if global_b_values.size
    else 0.0
)
thresholds = {}
for sample_id in pd.Index(sample_values).unique():
    mask = sample_values.eq(sample_id).to_numpy(dtype=bool) & is_b_cell
    values = cr2_cpm[mask]
    thresholds[str(sample_id)] = (
        float(np.quantile(values, CR2_POSITIVE_QUANTILE))
        if values.size
        else global_threshold
    )

threshold_per_cell = sample_values.astype(str).map(thresholds).to_numpy(dtype=float)
is_cr2_positive = cr2_cpm > threshold_per_cell

rows = []
for tls_id in pd.Index(tls_values[tls_valid]).unique():
    in_tls = tls_values.eq(tls_id).to_numpy(dtype=bool)
    in_tls_b = in_tls & is_b_cell
    n_b_cells = int(in_tls_b.sum())
    n_cr2_positive = int((in_tls_b & is_cr2_positive).sum())
    positive_fraction = n_cr2_positive / n_b_cells if n_b_cells else 0.0
    is_mature = (
        n_b_cells >= MIN_B_CELLS_PER_TLS
        and positive_fraction >= MIN_CR2_POSITIVE_FRACTION
    )
    trajectory_group = choose_mode(trajectory_labels.loc[in_tls])
    final_group = "Mature" if is_mature else trajectory_group
    sample_mode = choose_mode(sample_values.loc[in_tls])
    rows.append(
        {
            "tls_id": tls_id,
            "sample": sample_mode,
            "n_cells_in_tls": int(in_tls.sum()),
            "n_B_cells_in_tls": n_b_cells,
            "sample_CR2_Q75_CPM_threshold": thresholds.get(str(sample_mode), np.nan),
            "n_CR2_positive_B_cells": n_cr2_positive,
            "fraction_CR2_positive_B_cells": positive_fraction,
            "mature_by_CR2": bool(is_mature),
            "trajectory_group": trajectory_group,
            "final_group": final_group,
        }
    )

tls_metrics = pd.DataFrame(rows).set_index("tls_id")
tls_metrics.to_csv(CR2_METRICS_PATH, sep="	")
tls_metrics[
    ["sample", "trajectory_group", "mature_by_CR2", "final_group"]
].to_csv(FINAL_CLASSIFICATION_PATH, sep="	")

print("Final unique-TLS counts:")
print(tls_metrics["final_group"].value_counts(dropna=False))
print(f"Saved final classification: {FINAL_CLASSIFICATION_PATH}")

if WRITE_ANNOTATED_H5AD:
    final_lookup = tls_metrics["final_group"]
    final_labels = tls_values.map(final_lookup).astype("string")
    final_labels.loc[~tls_valid] = pd.NA
    adata.obs["tls_reference_group_before_CR2"] = pd.Categorical(
        trajectory_labels,
        categories=["Conforming", "Deviating"],
    )
    adata.obs["tls_reference_group"] = pd.Categorical(
        final_labels,
        categories=["Mature", "Conforming", "Deviating"],
    )
    adata.write_h5ad(FINAL_H5AD_PATH, compression="gzip")
    print(f"Saved annotated AnnData: {FINAL_H5AD_PATH}")


## 7. Classification summary

The first panel shows the mean ten-position profiles used for the two trajectory groups.
The second panel reports unique TLS-like structures after CR2 integration.


In [ ]:
plot_profiles = pre_mfuzz.join(trajectory[["Group"]], how="inner")
state_colors = {
    "Conforming": "#2878B5",
    "Deviating": "#C7473A",
    "Mature": "#3A8D5D",
    "Unassigned": "#7A7A7A",
}

fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.8), gridspec_kw={"width_ratios": [1.7, 1]})
for group in ("Conforming", "Deviating"):
    values = plot_profiles.loc[plot_profiles["Group"] == group, range(10)].to_numpy(float)
    for row in values:
        axes[0].plot(range(10), row, color=state_colors[group], alpha=0.10, linewidth=0.7)
    axes[0].plot(
        range(10),
        values.mean(axis=0),
        color=state_colors[group],
        linewidth=2.2,
        label=group,
    )
axes[0].set(xlabel="Alignment profile position", ylabel="Aggregated correlation profile")
axes[0].legend(frameon=False)

count_order = ["Mature", "Conforming", "Deviating", "Unassigned"]
count_values = tls_metrics["final_group"].fillna("Unassigned")
counts = count_values.value_counts().reindex(count_order, fill_value=0)
axes[1].bar(
    counts.index,
    counts.values,
    color=[state_colors[state] for state in counts.index],
    width=0.68,
)
for index, value in enumerate(counts.values):
    axes[1].text(index, value, str(value), ha="center", va="bottom")
axes[1].set(xlabel="Final TLS state", ylabel="Number of TLS-like structures")
axes[1].tick_params(axis="x", rotation=30)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FINAL_DIR / "TLS_classification_summary.pdf", bbox_inches="tight")
fig.savefig(FINAL_DIR / "TLS_classification_summary.png", dpi=300, bbox_inches="tight")
plt.show()


## Output files

The workflow writes reusable, TLS-level files under the configured output directory:

- `alignment/reference_pseudocell_qc.tsv`
- `alignment/spatial_B_to_reference_spearman.tsv.gz`
- `alignment/TLS_alignment_top2_sum.tsv`
- `alignment/TLS_alignment_qc.tsv`
- `alignment/pre_mfuzz.tsv`
- `mfuzz/trajectory_groups.tsv`
- `mfuzz/mfuzz_cluster_definitions.tsv`
- `final/TLS_CR2_metrics.tsv`
- `final/TLS_final_classification.tsv`
- `final/TLS_classification_summary.pdf`

The annotated H5AD is optional because the cell-level file can be large. Set
`TLS_WRITE_H5AD=1` before execution to enable it.
